# Data Rescue Kit
## MAD 2502 — Applied Data

**Big idea:** When data behave strangely, diagnose the type of problem before trying random fixes.

This notebook is a reference rather than a linear lecture, but every repair follows the same cycle:

> **Understand → Predict → Run → Change → Use → Practice → Connect**


In [ ]:
from pathlib import Path

import pandas as pd


## 1. Understand

Most data problems in this course fall into a few categories:

- Python cannot find the file;
- a column has the wrong type;
- values are missing;
- names or dates are inconsistent;
- rows are duplicated;
- JSON is nested;
- a filter unexpectedly returns no rows.

The first question is not *How do I fix it?* It is *What kind of problem is this?*


## 2. Predict

Predict the result before running each diagnosis:

```python
Path.cwd()
df.dtypes
df.isna().sum()
df.duplicated().sum()
```

Which one helps with a missing file? Which one helps with text stored in a numeric column? Which one detects repeated rows?


In [ ]:
print("working folder:", Path.cwd())
print("files here:")
for path in sorted(Path.cwd().iterdir()):
    print(" -", path.name)


## 3. Run

Create a deliberately messy table and diagnose it before changing anything.


In [ ]:
messy = pd.DataFrame(
    {
        "Event Name": ["A", "B", "B", "C"],
        "Cost": ["$1,200", "$0", "$0", None],
        "Attendance": ["150", "85", "85", "unknown"],
        "Date": ["2026-10-01", "10/05/2026", "10/05/2026", "bad date"],
    }
)

display(messy)
print(messy.dtypes)
print("missing values:")
print(messy.isna().sum())
print("duplicate rows:", messy.duplicated().sum())


The output tells us several different things are wrong. A single magic cleaning command would hide those distinctions.


## 4. Change

Fix one problem at a time, and verify each fix.


In [ ]:
clean = messy.copy()

clean.columns = [
    name.strip().lower().replace(" ", "_")
    for name in clean.columns
]

clean["cost"] = pd.to_numeric(
    clean["cost"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce",
)

clean["attendance"] = pd.to_numeric(
    clean["attendance"],
    errors="coerce",
)

clean["date"] = pd.to_datetime(
    clean["date"],
    errors="coerce",
)

clean = clean.drop_duplicates().reset_index(drop=True)

display(clean)
print(clean.dtypes)
print(clean.isna().sum())


Notice that `errors="coerce"` does not invent a number. Values that cannot be interpreted become missing, which lets you inspect and decide what to do next.


## 5. Use

### Empty DataFrames

If a filter unexpectedly returns nothing, check the values and types used in the condition before assuming the dataset is empty.


In [ ]:
expensive = clean[clean["cost"] > 5000]

print("rows returned:", len(expensive))
print("cost range:", clean["cost"].min(), "to", clean["cost"].max())
print("unique costs:", sorted(clean["cost"].dropna().unique()))


### Nested JSON

APIs may place dictionaries inside dictionaries. Inspect one record before trying to build the final table.


In [ ]:
records = [
    {"name": "A", "location": {"city": "Gainesville", "state": "FL"}},
    {"name": "B", "location": {"city": "Orlando", "state": "FL"}},
]

print(records[0])
flat = pd.json_normalize(records)
display(flat)


## 6. Practice

For each problem, identify the diagnosis before writing the fix.

1. Python says a CSV file does not exist. What should you inspect first?
2. A column containing `$2,500` has dtype `object`. Clean it.
3. A column uses `N/A`, blanks, and `unknown`. Convert valid values to numbers without inventing zeros.
4. A filter returns zero rows. List three checks you would make.
5. A JSON response has dictionaries nested inside each record. Flatten it.
6. Explain why dropping every row with any missing value can be a bad default.


## 7. Connect

A repeatable debugging routine is:

1. Read the error or unexpected result.
2. Inspect the smallest relevant object.
3. Check type, shape, column names, and a few values.
4. Make one small fix.
5. Verify that the fix changed what you expected.
6. Only then continue the larger analysis.

This is especially important in the capstone: cleaning decisions belong in your project journal because they are part of the analysis, not invisible setup work.
